In [ ]:
from datasets import load_dataset
import json
import torch
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import numpy as np
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import umap

In [ ]:
delta_sec = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00, 4.00, 5.00]

In [ ]:
dataset = load_dataset("CAiRE/ASCEND")

with open("/kaggle/input/datasets/mallikakalita/cs-times/code_switch_timestamps.json", "r") as f:
    code_switch_timestamps = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
wav2vec2_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec2_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav2vec2_model.eval()

In [ ]:
def get_switch_ground_truth(split, idx, delta_sec):

    sample = dataset[split][idx]

    switches = []
    for item in code_switch_timestamps[split]:
        if item["index"] == idx:
            for switch in item["switches"]:
                switches.append(switch["switch"])
            break

    waveform = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
    sample_rate = sample["audio"]["sampling_rate"]

    inputs = wav2vec2_processor(
        waveform.squeeze(0).numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt"
    )
    
    input_values = inputs.input_values.to(device)
    
    with torch.inference_mode():
        outputs = wav2vec2_model(
            input_values,
            output_hidden_states=False
        )
    
    wv_num_frames = outputs.last_hidden_state.size(1)
    samples_per_frame = input_values.size(1) / wv_num_frames
    seconds_per_frame = samples_per_frame / sample_rate
    
    frame_times = np.arange(wv_num_frames) * seconds_per_frame
    
    y = {}
    
    for delta in delta_sec:
        
        y[delta] = []
        
        for frame_time in frame_times:
        
            future_switches = [
                switch for switch in switches
                if switch > frame_time
            ]
        
            if len(future_switches) == 0:
                y[delta].append(0)
                continue
        
            next_switch = min(future_switches)
        
            if next_switch - frame_time <= delta:
                y[delta].append(1)
            else:
                y[delta].append(0)
        
        y[delta] = np.array(y[delta])
        
    return y

In [ ]:
ground_truth = {}
failures = []

for split in dataset.keys():

    ground_truth[split] = []

    for idx, sample in enumerate(tqdm(dataset[split], desc=split)):

        if sample["language"] != "mixed":
            continue

        if "[UNK]" in sample["transcription"]:
            continue

        try:
            y = get_switch_ground_truth(split, idx, delta_sec)

            ground_truth[split].append({
                "index": idx,
                "y": y
            })

        except Exception as e:
            failures.append({
                "split": split,
                "index": idx,
                "error": str(e)
            })

In [ ]:
def extract_layer_features(sample, layer):
    waveform = torch.tensor(sample["audio"]["array"], dtype=torch.float32).unsqueeze(0)
    sample_rate = sample["audio"]["sampling_rate"]

    inputs = wav2vec2_processor(
        waveform.squeeze(0).numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt"
    )
        
    input_values = inputs.input_values.to(device)
    
    with torch.inference_mode():
        outputs = wav2vec2_model(
            input_values,
            output_hidden_states=True
        )
    
    hidden_states = outputs.hidden_states
    
    features = hidden_states[layer].squeeze(0)

    X = features.cpu().numpy()

    return X

In [ ]:
def evaluate_layer(layer, dataset, ground_truth, delta):
    print("Layer: ", layer)
    
    X = {}
    y = {}

    for split in dataset.keys():
        
        X[split] = []
        y[split] = []

        for gt in ground_truth[split]:
            
            idx = gt["index"]
            sample = dataset[split][idx]
            y_sample = gt["y"][delta]
            
            X_sample = extract_layer_features(sample, layer)
            if X_sample.shape[0] != len(y_sample):
                raise ValueError(f"X and y shape mismatch in {split}, index {idx}")
                
            X[split].append(X_sample)
            y[split].append(y_sample)

        X[split] = np.concatenate(X[split], axis=0)
        y[split] = np.concatenate(y[split], axis=0)

        print(f"{split}: X = {X[split].shape}, y = {y[split].shape}")

    X_train = X["train"]
    y_train = y["train"]

    # X_val = X["validation"]
    # y_val = y["validation"]

    X_test = X["test"]
    y_test = y["test"]

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )

    model.fit(X_train, y_train)
    print("Number of iterations:", model[-1].n_iter_)

    #val_prob = model.predict_proba(X_val)[:, 1]

    test_prob = model.predict_proba(X_test)[:, 1]

    test_metrics = {
        "roc_auc": float(roc_auc_score(y_test, test_prob)),
        "ap": float(average_precision_score(y_test, test_prob)),
        "log_loss": float(log_loss(y_test, test_prob)),
        "brier_score": float(brier_score_loss(y_test, test_prob))
    }

    print("\nTest metrics:")
    for metric, value in test_metrics.items():
        print(f"{metric}: {value:.3f}")

    prob_true, prob_pred = calibration_curve(
        y_test,
        test_prob,
        n_bins=10
    )
    
    plt.figure(figsize=(6, 6))
    
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.plot(prob_pred, prob_true, marker="o")
    
    plt.xlabel("Mean predicted probability of code-switch within ∆")
    plt.ylabel("Observed fraction of frames with code-switch within ∆")
    plt.title(f"Calibration Curve — Layer {layer}")
    
    plt.show()

    max_umap_samples = 10000

    if len(X_test) > max_umap_samples:

        rng = np.random.default_rng(42)

        umap_indices = rng.choice(
            len(X_test),
            size=max_umap_samples,
            replace=False
        )

        X_umap = X_test[umap_indices]
        y_umap = y_test[umap_indices]

    else:

        X_umap = X_test
        y_umap = y_test

    reducer = umap.UMAP(
        n_components=2,
        random_state=42
    )

    umap_coordinates = reducer.fit_transform(X_umap)

    plt.figure(figsize=(8, 6))
    
    plt.scatter(
        umap_coordinates[:, 0],
        umap_coordinates[:, 1],
        c=(y_umap == 1),
        s=5
    )
    
    plt.title(f"UMAP - Layer {layer}, ∆ = {delta * 1000:.0f} ms")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.show()

    results = {
        "layer": layer,
        "delta_sec": delta,
        "metrics": test_metrics
    }
    
    return results

In [ ]:
def evaluate_all_layers(dataset, ground_truth, delta):
    
    results = {}
    
    num_layers = wav2vec2_model.config.num_hidden_layers + 1
    
    for layer in range(num_layers):
        result = evaluate_layer(layer, dataset, ground_truth, delta)
        results[layer] = result

    return results

In [ ]:
results = {}

for delta in delta_sec:
    print(f"Delta: {delta * 1000:.0f} ms")
    results[delta] = evaluate_all_layers(dataset, ground_truth, delta)

In [ ]:
with open("outputs/fs2_pre_switch_results.json", "w") as f:
    json.dump(results, f, indent=2)